# 05 — SOP / RAG Retrieval Foundation

This notebook builds the retrieval foundation for analyst-facing SOP and policy guidance.

Safety boundary:

- Deterministic triage remains authoritative for claim routing.
- Retrieved content may provide supporting SOP guidance only.
- Retrieval must not override eligibility, outcome, triggering rule, or controlled customer follow-up.
- Evaluation assets and staging copies are excluded from the runtime retrieval corpus.
- Every retrieved result must retain source-document and chunk-level provenance.

In [1]:
# ============================================================
# Step 1: Runtime Knowledge-Base Inventory
# ============================================================

from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

knowledge_base_dir = project_root / "data" / "knowledge_base"
staging_dir = project_root / "data" / "staging"
evaluation_dir = project_root / "data" / "evaluation"

print("Project root:", project_root)
print("Knowledge-base directory:", knowledge_base_dir)
print("Knowledge-base exists:", knowledge_base_dir.exists())

knowledge_base_files = sorted(
    path
    for path in knowledge_base_dir.rglob("*")
    if path.is_file()
)

inventory_rows = []

for file_path in knowledge_base_files:
    inventory_rows.append(
        {
            "relative_path": str(file_path.relative_to(project_root)),
            "file_name": file_path.name,
            "suffix": file_path.suffix.lower(),
            "size_bytes": file_path.stat().st_size,
        }
    )

knowledge_base_inventory = pd.DataFrame(inventory_rows)

print("\nKnowledge-base file count:", len(knowledge_base_inventory))

if not knowledge_base_inventory.empty:
    display(knowledge_base_inventory)
else:
    print("No knowledge-base files found.")

print("\nSafety check:")
print("Staging directory excluded from retrieval corpus:", staging_dir.exists())
print("Evaluation directory excluded from retrieval corpus:", evaluation_dir.exists())

Project root: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage
Knowledge-base directory: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/data/knowledge_base
Knowledge-base exists: True

Knowledge-base file count: 7


,relative_path,file_name,suffix,size_bytes
0,data/knowledge_base/KB-001_claims_triage_sop_v...,KB-001_claims_triage_sop_v1.md,.md,3480
1,data/knowledge_base/KB-002_evidence_review_gui...,KB-002_evidence_review_guide_v1.md,.md,3167
2,data/knowledge_base/KB-003_follow_up_communica...,KB-003_follow_up_communication_guide_v1.md,.md,2813
3,data/knowledge_base/KB-004_manual_review_escal...,KB-004_manual_review_escalation_sop_v1.md,.md,2590
4,data/knowledge_base/KB-005_decision_explanatio...,KB-005_decision_explanation_audit_guide_v1.md,.md,2261
5,data/knowledge_base/KB-006_safety_privacy_scop...,KB-006_safety_privacy_scope_boundaries_v1.md,.md,2716
6,data/knowledge_base/KB-007_claims_triage_gloss...,KB-007_claims_triage_glossary_v1.md,.md,2101



Safety check:
Staging directory excluded from retrieval corpus: True
Evaluation directory excluded from retrieval corpus: True


In [2]:
# ============================================================
# Step 0: Notebook Bootstrap
# ============================================================

import sys
from pathlib import Path


def find_project_root(start_path: Path) -> Path:
    """Locate the project root by finding src/data_loader.py."""
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src" / "data_loader.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing src/data_loader.py."
    )


project_root = find_project_root(Path.cwd().resolve())

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Import path added:", sys.path[0])

assert (project_root / "src" / "data_loader.py").is_file()

print("Notebook bootstrap completed successfully.")

Project root: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage
Import path added: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage
Notebook bootstrap completed successfully.


In [3]:
# ============================================================
# Step 1B: Preview Text-Based Knowledge-Base Documents
# ============================================================

TEXT_SUFFIXES = {".md", ".txt", ".csv", ".json"}

text_files = [
    path
    for path in knowledge_base_files
    if path.suffix.lower() in TEXT_SUFFIXES
]

for file_path in text_files:
    print("\n" + "=" * 90)
    print("FILE:", file_path.relative_to(project_root))
    print("=" * 90)

    try:
        content = file_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
        print(content[:4000])

        if len(content) > 4000:
            print("\n[Preview truncated]")
    except Exception as error:
        print(f"Could not read file: {error}")


FILE: data/knowledge_base/KB-001_claims_triage_sop_v1.md
# Claims Triage Standard Operating Procedure (SOP)

**Document ID:** KB-001  
**Version:** 1.0  
**Effective date:** 2026-06-23  
**Applies to:** DeviceProtect smartphone claims triage MVP  
**Purpose:** Describe the operational sequence for a triage agent. This SOP does not define product coverage or override the Policy Rule Catalog.

## 1. Objective

Produce a traceable triage recommendation using authoritative structured sources, then route the case to standard processing, follow-up, or an analyst. The agent supports operations; it does not approve payment, deny a claim finally, determine fraud, or make a binding customer commitment.

## 2. Authoritative source order

Use sources according to their role:

1. **Policy Eligibility Lookup (Item #4):** policy, enrolment, coverage-window, and covered-device facts.
2. **Plan Master and Coverage Matrix (Item #2):** plan attributes and general category coverage.
3. **Policy Rule Cata

## Step 2 — RAG Document Registry

This step creates an explicit runtime allow-list for the initial retrieval corpus.

The registry separates documents that may support analyst-facing retrieval from documents that remain controlled by deterministic tools, static safety controls, or query-normalisation logic.

In [4]:
# ============================================================
# Step 2: Create Controlled RAG Document Registry
# ============================================================

from pathlib import Path
import pandas as pd

registry_path = (
    project_root
    / "data"
    / "runtime"
    / "reference"
    / "rag_document_registry_v1.csv"
)

rag_document_registry = pd.DataFrame(
    [
        {
            "document_id": "KB-001",
            "relative_path": (
                "data/knowledge_base/"
                "KB-001_claims_triage_sop_v1.md"
            ),
            "document_type": "TRIAGE_SOP",
            "retrieval_status": "INCLUDED",
            "retrieval_scope": "ANALYST_GUIDANCE",
            "authority_role": "OPERATIONAL_GUIDANCE_ONLY",
            "priority": 1,
            "reason": (
                "Core triage process guidance. Structured policy and rule "
                "sources remain authoritative."
            ),
        },
        {
            "document_id": "KB-002",
            "relative_path": (
                "data/knowledge_base/"
                "KB-002_evidence_review_guide_v1.md"
            ),
            "document_type": "EVIDENCE_GUIDE",
            "retrieval_status": "INCLUDED",
            "retrieval_scope": "ANALYST_GUIDANCE",
            "authority_role": "OPERATIONAL_GUIDANCE_ONLY",
            "priority": 2,
            "reason": (
                "Supports evidence-review explanation. Evidence profile "
                "requirements remain authoritative in structured data."
            ),
        },
        {
            "document_id": "KB-003",
            "relative_path": (
                "data/knowledge_base/"
                "KB-003_follow_up_communication_guide_v1.md"
            ),
            "document_type": "FOLLOW_UP_COMMUNICATION",
            "retrieval_status": "EXCLUDED",
            "retrieval_scope": "DETERMINISTIC_TOOL_ONLY",
            "authority_role": "APPROVED_COMMUNICATION_REFERENCE",
            "priority": 0,
            "reason": (
                "Customer follow-up is selected from the deterministic "
                "approved-question catalogue, not generated through RAG."
            ),
        },
        {
            "document_id": "KB-004",
            "relative_path": (
                "data/knowledge_base/"
                "KB-004_manual_review_escalation_sop_v1.md"
            ),
            "document_type": "MANUAL_REVIEW_SOP",
            "retrieval_status": "INCLUDED",
            "retrieval_scope": "ANALYST_GUIDANCE",
            "authority_role": "OPERATIONAL_GUIDANCE_ONLY",
            "priority": 2,
            "reason": (
                "Supports fact-based analyst escalation guidance for "
                "MANUAL_REVIEW outcomes."
            ),
        },
        {
            "document_id": "KB-005",
            "relative_path": (
                "data/knowledge_base/"
                "KB-005_decision_explanation_audit_guide_v1.md"
            ),
            "document_type": "EXPLANATION_AUDIT_GUIDE",
            "retrieval_status": "INCLUDED",
            "retrieval_scope": "ANALYST_GUIDANCE",
            "authority_role": "OPERATIONAL_GUIDANCE_ONLY",
            "priority": 2,
            "reason": (
                "Supports traceable analyst explanation and audit output. "
                "It must not be used for customer-facing final-decision text."
            ),
        },
        {
            "document_id": "KB-006",
            "relative_path": (
                "data/knowledge_base/"
                "KB-006_safety_privacy_scope_boundaries_v1.md"
            ),
            "document_type": "SAFETY_SCOPE_POLICY",
            "retrieval_status": "EXCLUDED",
            "retrieval_scope": "STATIC_SYSTEM_CONTROL",
            "authority_role": "NON_NEGOTIABLE_SAFETY_CONTROL",
            "priority": 0,
            "reason": (
                "Safety and privacy boundaries must be enforced by prompts, "
                "guardrails, and code rather than depending on retrieval."
            ),
        },
        {
            "document_id": "KB-007",
            "relative_path": (
                "data/knowledge_base/"
                "KB-007_claims_triage_glossary_v1.md"
            ),
            "document_type": "GLOSSARY",
            "retrieval_status": "REFERENCE_ONLY",
            "retrieval_scope": "QUERY_NORMALISATION",
            "authority_role": "TERMINOLOGY_REFERENCE",
            "priority": 0,
            "reason": (
                "Supports terminology normalisation but is not part of the "
                "initial primary retrieval corpus."
            ),
        },
    ]
)

registry_path.parent.mkdir(parents=True, exist_ok=True)

rag_document_registry.to_csv(
    registry_path,
    index=False,
)

print("Registry written to:", registry_path)
print("\nRegistry preview:")
display(rag_document_registry)

included_documents = rag_document_registry.loc[
    rag_document_registry["retrieval_status"].eq("INCLUDED")
].copy()

print("\nInitial retrieval corpus:")
display(
    included_documents[
        [
            "document_id",
            "document_type",
            "priority",
            "relative_path",
        ]
    ]
)

assert len(rag_document_registry) == 7
assert len(included_documents) == 4

assert set(included_documents["document_id"]) == {
    "KB-001",
    "KB-002",
    "KB-004",
    "KB-005",
}

assert "KB-003" not in set(included_documents["document_id"])
assert "KB-006" not in set(included_documents["document_id"])
assert "KB-007" not in set(included_documents["document_id"])

print(
    "\nValidation passed: the v1 retrieval corpus contains only the "
    "four approved analyst-guidance documents."
)

Registry written to: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/data/runtime/reference/rag_document_registry_v1.csv

Registry preview:


,document_id,relative_path,document_type,retrieval_status,retrieval_scope,authority_role,priority,reason
0,KB-001,data/knowledge_base/KB-001_claims_triage_sop_v...,TRIAGE_SOP,INCLUDED,ANALYST_GUIDANCE,OPERATIONAL_GUIDANCE_ONLY,1,Core triage process guidance. Structured polic...
1,KB-002,data/knowledge_base/KB-002_evidence_review_gui...,EVIDENCE_GUIDE,INCLUDED,ANALYST_GUIDANCE,OPERATIONAL_GUIDANCE_ONLY,2,Supports evidence-review explanation. Evidence...
2,KB-003,data/knowledge_base/KB-003_follow_up_communica...,FOLLOW_UP_COMMUNICATION,EXCLUDED,DETERMINISTIC_TOOL_ONLY,APPROVED_COMMUNICATION_REFERENCE,0,Customer follow-up is selected from the determ...
3,KB-004,data/knowledge_base/KB-004_manual_review_escal...,MANUAL_REVIEW_SOP,INCLUDED,ANALYST_GUIDANCE,OPERATIONAL_GUIDANCE_ONLY,2,Supports fact-based analyst escalation guidanc...
4,KB-005,data/knowledge_base/KB-005_decision_explanatio...,EXPLANATION_AUDIT_GUIDE,INCLUDED,ANALYST_GUIDANCE,OPERATIONAL_GUIDANCE_ONLY,2,Supports traceable analyst explanation and aud...
5,KB-006,data/knowledge_base/KB-006_safety_privacy_scop...,SAFETY_SCOPE_POLICY,EXCLUDED,STATIC_SYSTEM_CONTROL,NON_NEGOTIABLE_SAFETY_CONTROL,0,Safety and privacy boundaries must be enforced...
6,KB-007,data/knowledge_base/KB-007_claims_triage_gloss...,GLOSSARY,REFERENCE_ONLY,QUERY_NORMALISATION,TERMINOLOGY_REFERENCE,0,Supports terminology normalisation but is not ...



Initial retrieval corpus:


,document_id,document_type,priority,relative_path
0,KB-001,TRIAGE_SOP,1,data/knowledge_base/KB-001_claims_triage_sop_v...
1,KB-002,EVIDENCE_GUIDE,2,data/knowledge_base/KB-002_evidence_review_gui...
3,KB-004,MANUAL_REVIEW_SOP,2,data/knowledge_base/KB-004_manual_review_escal...
4,KB-005,EXPLANATION_AUDIT_GUIDE,2,data/knowledge_base/KB-005_decision_explanatio...



Validation passed: the v1 retrieval corpus contains only the four approved analyst-guidance documents.


In [5]:
# ============================================================
# Step 3A: Refresh Runtime Data with RAG Registry
# ============================================================

import importlib

import src.data_loader as data_loader

importlib.reload(data_loader)

data = data_loader.load_runtime_data()

print("RAG registry rows:", len(data["rag_document_registry"]))
display(data["rag_document_registry"])

RAG registry rows: 7


,document_id,relative_path,document_type,retrieval_status,retrieval_scope,authority_role,priority,reason
0,KB-001,data/knowledge_base/KB-001_claims_triage_sop_v...,TRIAGE_SOP,INCLUDED,ANALYST_GUIDANCE,OPERATIONAL_GUIDANCE_ONLY,1,Core triage process guidance. Structured polic...
1,KB-002,data/knowledge_base/KB-002_evidence_review_gui...,EVIDENCE_GUIDE,INCLUDED,ANALYST_GUIDANCE,OPERATIONAL_GUIDANCE_ONLY,2,Supports evidence-review explanation. Evidence...
2,KB-003,data/knowledge_base/KB-003_follow_up_communica...,FOLLOW_UP_COMMUNICATION,EXCLUDED,DETERMINISTIC_TOOL_ONLY,APPROVED_COMMUNICATION_REFERENCE,0,Customer follow-up is selected from the determ...
3,KB-004,data/knowledge_base/KB-004_manual_review_escal...,MANUAL_REVIEW_SOP,INCLUDED,ANALYST_GUIDANCE,OPERATIONAL_GUIDANCE_ONLY,2,Supports fact-based analyst escalation guidanc...
4,KB-005,data/knowledge_base/KB-005_decision_explanatio...,EXPLANATION_AUDIT_GUIDE,INCLUDED,ANALYST_GUIDANCE,OPERATIONAL_GUIDANCE_ONLY,2,Supports traceable analyst explanation and aud...
5,KB-006,data/knowledge_base/KB-006_safety_privacy_scop...,SAFETY_SCOPE_POLICY,EXCLUDED,STATIC_SYSTEM_CONTROL,NON_NEGOTIABLE_SAFETY_CONTROL,0,Safety and privacy boundaries must be enforced...
6,KB-007,data/knowledge_base/KB-007_claims_triage_gloss...,GLOSSARY,REFERENCE_ONLY,QUERY_NORMALISATION,TERMINOLOGY_REFERENCE,0,Supports terminology normalisation but is not ...


## Step 3C — Provenance-Preserving Corpus Construction

This step converts only registry-approved Markdown documents into heading-aware retrieval chunks.

Each chunk keeps its document ID, version, section title, source path, and content hash. References sections are excluded because they provide citation pointers rather than analyst guidance.

In [6]:
# ============================================================
# Step 3C: Build and Inspect Controlled RAG Corpus
# ============================================================

from src.rag.corpus_builder import build_rag_corpus

rag_corpus = build_rag_corpus(
    registry=data["rag_document_registry"],
)

print("Total chunks:", len(rag_corpus))
print(
    "Documents represented:",
    sorted(rag_corpus["document_id"].unique()),
)

print("\nChunk count by document:")
display(
    rag_corpus.groupby(
        ["document_id", "document_title"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "chunk_count"})
)

print("\nCorpus provenance preview:")
display(
    rag_corpus[
        [
            "chunk_id",
            "document_id",
            "document_version",
            "document_type",
            "section_title",
            "chunk_char_count",
            "source_relative_path",
        ]
    ]
)

print("\nSample chunk:")
sample_chunk = rag_corpus.iloc[0]

print("Chunk ID:", sample_chunk["chunk_id"])
print("Document:", sample_chunk["document_title"])
print("Section:", sample_chunk["section_title"])
print("\n", sample_chunk["chunk_text"])

assert set(rag_corpus["document_id"]) == {
    "KB-001",
    "KB-002",
    "KB-004",
    "KB-005",
}

assert not set(rag_corpus["document_id"]).intersection(
    {"KB-003", "KB-006", "KB-007"}
)

assert rag_corpus["chunk_id"].is_unique
assert rag_corpus["chunk_text"].str.len().gt(0).all()
assert rag_corpus["chunk_sha256"].str.len().eq(64).all()

assert not rag_corpus["section_title"].str.casefold().eq(
    "references"
).any()

assert rag_corpus["source_relative_path"].str.startswith(
    "data/knowledge_base/"
).all()

print(
    "\nValidation passed: only approved knowledge-base content was "
    "converted into provenance-preserving retrieval chunks."
)

Total chunks: 21
Documents represented: ['KB-001', 'KB-002', 'KB-004', 'KB-005']

Chunk count by document:


,document_id,document_title,chunk_count
0,KB-001,Claims Triage Standard Operating Procedure (SOP),6
1,KB-002,Evidence Review Guide,5
2,KB-004,Manual Review and Escalation SOP,5
3,KB-005,Decision Explanation and Audit Guide,5



Corpus provenance preview:


,chunk_id,document_id,document_version,document_type,section_title,chunk_char_count,source_relative_path
0,KB-001::S01,KB-001,1.0,TRIAGE_SOP,Document Overview,371,data/knowledge_base/KB-001_claims_triage_sop_v...
1,KB-001::S02,KB-001,1.0,TRIAGE_SOP,Objective,368,data/knowledge_base/KB-001_claims_triage_sop_v...
2,KB-001::S03,KB-001,1.0,TRIAGE_SOP,Authoritative source order,916,data/knowledge_base/KB-001_claims_triage_sop_v...
3,KB-001::S04,KB-001,1.0,TRIAGE_SOP,Standard triage flow,1205,data/knowledge_base/KB-001_claims_triage_sop_v...
4,KB-001::S05,KB-001,1.0,TRIAGE_SOP,Handling incomplete information,353,data/knowledge_base/KB-001_claims_triage_sop_v...
5,KB-001::S06,KB-001,1.0,TRIAGE_SOP,Prohibited actions,436,data/knowledge_base/KB-001_claims_triage_sop_v...
6,KB-002::S01,KB-002,1.0,EVIDENCE_GUIDE,Document Overview,347,data/knowledge_base/KB-002_evidence_review_gui...
7,KB-002::S02,KB-002,1.0,EVIDENCE_GUIDE,Evidence-review principles,601,data/knowledge_base/KB-002_evidence_review_gui...
8,KB-002::S03,KB-002,1.0,EVIDENCE_GUIDE,Evidence profiles,653,data/knowledge_base/KB-002_evidence_review_gui...
9,KB-002::S04,KB-002,1.0,EVIDENCE_GUIDE,Evidence status handling,553,data/knowledge_base/KB-002_evidence_review_gui...



Sample chunk:
Chunk ID: KB-001::S01
Document: Claims Triage Standard Operating Procedure (SOP)
Section: Document Overview

 Document: Claims Triage Standard Operating Procedure (SOP)
Section: Document Overview

**Document ID:** KB-001  
**Version:** 1.0  
**Effective date:** 2026-06-23  
**Applies to:** DeviceProtect smartphone claims triage MVP  
**Purpose:** Describe the operational sequence for a triage agent. This SOP does not define product coverage or override the Policy Rule Catalog.

Validation passed: only approved knowledge-base content was converted into provenance-preserving retrieval chunks.


In [7]:
# ============================================================
# Step 4A: Check TF-IDF Retrieval Dependency
# ============================================================

try:
    import sklearn

    print("scikit-learn version:", sklearn.__version__)
    print("TF-IDF retrieval dependency is available.")
except ModuleNotFoundError:
    print(
        "scikit-learn is not installed in this environment. "
        "Install it in the active dpclaims environment before continuing."
    )

scikit-learn version: 1.9.0
TF-IDF retrieval dependency is available.


## Step 4C — Lexical TF-IDF Retrieval Baseline

This step creates a transparent lexical retrieval baseline over the approved analyst-guidance corpus.

Retrieved content remains non-authoritative. It supports analyst guidance and traceability only; deterministic triage continues to control routing, rule application, and customer follow-up selection.

In [8]:
# ============================================================
# Step 4C: Build and Inspect Lexical Retrieval Baseline
# ============================================================

from pprint import pprint

from src.rag.lexical_retriever import ControlledLexicalRetriever


lexical_retriever = ControlledLexicalRetriever(rag_corpus)

print("Indexed chunk count:", lexical_retriever.corpus_size)

evidence_result = lexical_retriever.retrieve(
    query=(
        "liquid damage claim missing required photo "
        "evidence follow-up guidance"
    ),
    top_k=3,
)

manual_review_result = lexical_retriever.retrieve(
    query=(
        "device identifier mismatch manual review "
        "escalation packet analyst guidance"
    ),
    top_k=3,
)

print("\nEvidence-guidance retrieval:")
for result in evidence_result["results"]:
    print(
        f"\nRank {result['rank']} | "
        f"Score {result['relevance_score']} | "
        f"{result['chunk_id']} | "
        f"{result['section_title']}"
    )
    print(result["chunk_text"][:700])

print("\nManual-review retrieval:")
for result in manual_review_result["results"]:
    print(
        f"\nRank {result['rank']} | "
        f"Score {result['relevance_score']} | "
        f"{result['chunk_id']} | "
        f"{result['section_title']}"
    )
    print(result["chunk_text"][:700])

assert evidence_result["retrieval_status"] == "RESULTS_FOUND"
assert manual_review_result["retrieval_status"] == "RESULTS_FOUND"

assert any(
    result["document_id"] == "KB-002"
    for result in evidence_result["results"]
)

assert any(
    result["document_id"] == "KB-004"
    for result in manual_review_result["results"]
)

assert all(
    result["source_reference"]["source_relative_path"].startswith(
        "data/knowledge_base/"
    )
    for result in (
        evidence_result["results"]
        + manual_review_result["results"]
    )
)

print(
    "\nValidation passed: lexical retrieval returned provenance-preserving "
    "analyst guidance from the approved corpus."
)

Indexed chunk count: 21

Evidence-guidance retrieval:

Rank 1 | Score 0.233116 | KB-002::S03 | Evidence profiles
Document: Evidence Review Guide
Section: Evidence profiles

Use the evidence profile linked to the applicable claim category:

| Claim category | Evidence profile | Primary required evidence |
|---|---|---|
| SCREEN_DAMAGE | EVD-SCREEN-01 | Damage photo |
| ACCIDENTAL_DAMAGE | EVD-ACCIDENTAL-01 | Damage photo |
| LIQUID_DAMAGE | EVD-LIQUID-01 | Damage photo |
| MECHANICAL_BREAKDOWN | EVD-MECHANICAL-01 | Diagnostic report |
| THEFT | EVD-THEFT-01 | Police report reference |

Repair quotes are supporting evidence where available and may be useful for repair feasibility or high-cost review. They are not automatically a reason to proceed or decline.

Rank 2 | Score 0.129048 | KB-002::S02 | Evidence-review principles
Document: Evidence Review Guide
Section: Evidence-review principles

- Evidence validates a potentially eligible claim; it does not create coverage.
- A required doc

In [9]:
# ============================================================
# Step 4F: Check Semantic Retrieval Dependencies
# ============================================================

from importlib.metadata import version
from dotenv import load_dotenv
import os

load_dotenv(project_root / ".env")

packages = [
    "numpy",
    "scikit-learn",
    "openai",
]

for package_name in packages:
    print(f"{package_name}: {version(package_name)}")

print(
    "\nOpenAI API key configured:",
    bool(os.getenv("OPENAI_API_KEY")),
)

numpy: 2.4.6
scikit-learn: 1.9.0
openai: 2.44.0

OpenAI API key configured: True


## Step 5C — Live Semantic Retrieval Validation

This step creates an in-memory semantic index for the approved analyst-guidance corpus and compares semantic retrieval with the lexical TF-IDF baseline.

Only approved knowledge-base chunks are embedded. No claim records, customer narratives, staging content, or evaluation data are included.

In [10]:
# ============================================================
# Step 5C: Live Semantic vs Lexical Retrieval Comparison
# ============================================================

from dotenv import load_dotenv
from openai import OpenAI

from src.rag.semantic_retriever import (
    ControlledSemanticRetriever,
)

load_dotenv(project_root / ".env")

openai_client = OpenAI()

semantic_retriever = ControlledSemanticRetriever.from_openai(
    corpus=rag_corpus,
    embedding_model="text-embedding-3-small",
    client=openai_client,
)

print("Semantic index created successfully.")
print("Indexed chunks:", semantic_retriever.corpus_size)
print("Embedding model:", semantic_retriever.embedding_model)
print("Embedding dimension:", semantic_retriever.embedding_dimension)
print("Corpus fingerprint:", semantic_retriever.corpus_fingerprint)

evidence_query = (
    "liquid damage claim missing required photo "
    "evidence follow-up guidance"
)

manual_review_query = (
    "device identifier mismatch manual review "
    "escalation packet analyst guidance"
)

lexical_evidence = lexical_retriever.retrieve(
    query=evidence_query,
    top_k=3,
)

semantic_evidence = semantic_retriever.retrieve(
    query=evidence_query,
    top_k=3,
    min_relevance_score=0.20,
    client=openai_client,
)

lexical_manual_review = lexical_retriever.retrieve(
    query=manual_review_query,
    top_k=3,
)

semantic_manual_review = semantic_retriever.retrieve(
    query=manual_review_query,
    top_k=3,
    min_relevance_score=0.20,
    client=openai_client,
)


def display_ranked_results(label, result):
    print(f"\n{'=' * 90}")
    print(label)
    print("=" * 90)
    print("Status:", result["retrieval_status"])

    for item in result["results"]:
        print(
            f"\nRank {item['rank']} | "
            f"Score {item['relevance_score']} | "
            f"{item['chunk_id']} | "
            f"{item['section_title']}"
        )
        print(item["chunk_text"][:500])


display_ranked_results(
    "Lexical — Evidence Guidance",
    lexical_evidence,
)

display_ranked_results(
    "Semantic — Evidence Guidance",
    semantic_evidence,
)

display_ranked_results(
    "Lexical — Manual Review Guidance",
    lexical_manual_review,
)

display_ranked_results(
    "Semantic — Manual Review Guidance",
    semantic_manual_review,
)

assert semantic_retriever.corpus_size == 21
assert semantic_retriever.embedding_dimension > 0

assert semantic_evidence["retrieval_status"] == "RESULTS_FOUND"
assert semantic_manual_review["retrieval_status"] == "RESULTS_FOUND"

all_semantic_results = (
    semantic_evidence["results"]
    + semantic_manual_review["results"]
)

assert all(
    item["document_id"] in {"KB-001", "KB-002", "KB-004", "KB-005"}
    for item in all_semantic_results
)

assert all(
    item["source_reference"]["source_relative_path"].startswith(
        "data/knowledge_base/"
    )
    for item in all_semantic_results
)

assert all(
    item["section_title"].casefold()
    != "examples of targeted requests"
    for item in all_semantic_results
)

print(
    "\nValidation passed: semantic retrieval used only approved "
    "analyst-guidance chunks with retained provenance."
)

Semantic index created successfully.
Indexed chunks: 21
Embedding model: text-embedding-3-small
Embedding dimension: 1536
Corpus fingerprint: f73f827d3614e2987a7e976c687043f3fe9e2f5faeeb9c6555508089230ebf48

Lexical — Evidence Guidance
Status: RESULTS_FOUND

Rank 1 | Score 0.233116 | KB-002::S03 | Evidence profiles
Document: Evidence Review Guide
Section: Evidence profiles

Use the evidence profile linked to the applicable claim category:

| Claim category | Evidence profile | Primary required evidence |
|---|---|---|
| SCREEN_DAMAGE | EVD-SCREEN-01 | Damage photo |
| ACCIDENTAL_DAMAGE | EVD-ACCIDENTAL-01 | Damage photo |
| LIQUID_DAMAGE | EVD-LIQUID-01 | Damage photo |
| MECHANICAL_BREAKDOWN | EVD-MECHANICAL-01 | Diagnostic report |
| THEFT | EVD-THEFT-01 | Police report reference |

Repair quotes are sup

Rank 2 | Score 0.129048 | KB-002::S02 | Evidence-review principles
Document: Evidence Review Guide
Section: Evidence-review principles

- Evidence validates a potentially eligible c

In [11]:
# ============================================================
# Step 6B: Live Hybrid Retrieval Validation
# ============================================================

from src.rag.hybrid_retriever import ControlledHybridRetriever


hybrid_retriever = ControlledHybridRetriever(
    lexical_retriever=lexical_retriever,
    semantic_retriever=semantic_retriever,
)

hybrid_evidence = hybrid_retriever.retrieve(
    query=evidence_query,
    top_k=3,
    candidate_k=5,
    semantic_client=openai_client,
)

hybrid_manual_review = hybrid_retriever.retrieve(
    query=manual_review_query,
    top_k=3,
    candidate_k=5,
    semantic_client=openai_client,
)


def display_hybrid_results(label, result):
    print(f"\n{'=' * 90}")
    print(label)
    print("=" * 90)
    print("Status:", result["retrieval_status"])

    for item in result["results"]:
        print(
            f"\nRank {item['rank']} | "
            f"Fusion {item['fusion_score']} | "
            f"{item['chunk_id']} | "
            f"{item['section_title']}"
        )
        print("Methods:", item["retrieval_methods"])
        print("Method ranks:", item["method_ranks"])
        print(item["chunk_text"][:500])


display_hybrid_results(
    "Hybrid — Evidence Guidance",
    hybrid_evidence,
)

display_hybrid_results(
    "Hybrid — Manual Review Guidance",
    hybrid_manual_review,
)

assert hybrid_retriever.corpus_size == 21
assert hybrid_evidence["retrieval_status"] == "RESULTS_FOUND"
assert hybrid_manual_review["retrieval_status"] == "RESULTS_FOUND"

assert any(
    item["document_id"] == "KB-002"
    for item in hybrid_evidence["results"]
)

assert any(
    item["document_id"] == "KB-004"
    for item in hybrid_manual_review["results"]
)

assert all(
    "semantic" in item["retrieval_methods"]
    or "lexical" in item["retrieval_methods"]
    for item in (
        hybrid_evidence["results"]
        + hybrid_manual_review["results"]
    )
)

print(
    "\nValidation passed: hybrid RRF retrieval fused lexical and semantic "
    "analyst-guidance results while retaining chunk-level provenance."
)


Hybrid — Evidence Guidance
Status: RESULTS_FOUND

Rank 1 | Fusion 0.03278689 | KB-002::S03 | Evidence profiles
Methods: ['lexical', 'semantic']
Method ranks: {'lexical': 1, 'semantic': 1}
Document: Evidence Review Guide
Section: Evidence profiles

Use the evidence profile linked to the applicable claim category:

| Claim category | Evidence profile | Primary required evidence |
|---|---|---|
| SCREEN_DAMAGE | EVD-SCREEN-01 | Damage photo |
| ACCIDENTAL_DAMAGE | EVD-ACCIDENTAL-01 | Damage photo |
| LIQUID_DAMAGE | EVD-LIQUID-01 | Damage photo |
| MECHANICAL_BREAKDOWN | EVD-MECHANICAL-01 | Diagnostic report |
| THEFT | EVD-THEFT-01 | Police report reference |

Repair quotes are sup

Rank 2 | Fusion 0.03225806 | KB-002::S02 | Evidence-review principles
Methods: ['lexical', 'semantic']
Method ranks: {'lexical': 2, 'semantic': 2}
Document: Evidence Review Guide
Section: Evidence-review principles

- Evidence validates a potentially eligible claim; it does not create coverage.
- A required 

## Step 7 — Fixed Retrieval Evaluation Set

This step creates a frozen, curated evaluation set for SOP retrieval.

The query set is not loaded by the runtime data loader or retrieval index. It is used only to compare lexical, semantic, and hybrid retrieval quality.

After this point, do not alter this v1 set to improve reported results. Any future material change should use a new versioned evaluation set.

In [12]:
# ============================================================
# Step 7B: Create Fixed Retrieval Evaluation Set
# ============================================================

import hashlib
import json
from pathlib import Path

import pandas as pd


evaluation_dir = (
    project_root
    / "data"
    / "evaluation"
    / "retrieval"
)

evaluation_path = (
    evaluation_dir
    / "rag_retrieval_eval_v1.csv"
)

evaluation_rows = [
    {
        "query_id": "RAG-EVAL-001",
        "query_family": "EVIDENCE_PROFILE",
        "query_text": (
            "A claimant reports liquid exposure. Which proof is mandatory "
            "for this category?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-002::S03"]
        ),
        "rationale": (
            "Tests category-to-evidence-profile mapping without using the "
            "exact table wording."
        ),
    },
    {
        "query_id": "RAG-EVAL-002",
        "query_family": "EVIDENCE_STATUS",
        "query_text": (
            "What is the correct operational response to a required image "
            "that cannot be read?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-002::S04"]
        ),
        "rationale": (
            "Tests handling of unreadable mandatory evidence."
        ),
    },
    {
        "query_id": "RAG-EVAL-003",
        "query_family": "EVIDENCE_CONTRADICTION",
        "query_text": (
            "A repair document identifies a different handset from the "
            "policy record. How should that inconsistency be handled?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-002::S05"]
        ),
        "rationale": (
            "Tests neutral handling of evidence that conflicts with "
            "structured device facts."
        ),
    },
    {
        "query_id": "RAG-EVAL-004",
        "query_family": "EVIDENCE_PRINCIPLE",
        "query_text": (
            "Can a supporting repair quotation make an otherwise uncovered "
            "event eligible?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-002::S02"]
        ),
        "rationale": (
            "Tests the principle that evidence validates a potentially "
            "eligible claim but does not create coverage."
        ),
    },
    {
        "query_id": "RAG-EVAL-005",
        "query_family": "MANUAL_REVIEW_PACKET",
        "query_text": (
            "What details must accompany a case assigned for analyst review?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-004::S03"]
        ),
        "rationale": (
            "Tests required escalation-packet content."
        ),
    },
    {
        "query_id": "RAG-EVAL-006",
        "query_family": "MANUAL_REVIEW_WRITING",
        "query_text": (
            "How should an escalation note describe conflicting device "
            "identifiers without alleging dishonesty?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-004::S04"]
        ),
        "rationale": (
            "Tests neutral, fact-based escalation writing."
        ),
    },
    {
        "query_id": "RAG-EVAL-007",
        "query_family": "MANUAL_REVIEW_SCOPE",
        "query_text": (
            "What investigations or approvals are outside the triage "
            "agent's role after a review trigger?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-004::S05"]
        ),
        "rationale": (
            "Tests the boundary between triage support and analyst actions."
        ),
    },
    {
        "query_id": "RAG-EVAL-008",
        "query_family": "SOURCE_PRECEDENCE",
        "query_text": (
            "When structured sources disagree, which source order and "
            "routing approach should govern the triage?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-001::S03"]
        ),
        "rationale": (
            "Tests authoritative-source precedence and safe conflict routing."
        ),
    },
    {
        "query_id": "RAG-EVAL-009",
        "query_family": "MATERIAL_FOLLOW_UP",
        "query_text": (
            "Should the system request additional paperwork after a "
            "conclusive policy failure already determines the route?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-001::S05"]
        ),
        "rationale": (
            "Tests the rule that follow-up must be material to the next "
            "decision step."
        ),
    },
    {
        "query_id": "RAG-EVAL-010",
        "query_family": "TRIAGE_FLOW",
        "query_text": (
            "What operational sequence should be completed before choosing "
            "a triage disposition?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-001::S04"]
        ),
        "rationale": (
            "Tests the standard operational triage sequence."
        ),
    },
    {
        "query_id": "RAG-EVAL-011",
        "query_family": "EXPLANATION_FACT_TYPES",
        "query_text": (
            "Which categories of facts and uncertainty need to be "
            "distinguished in an auditable explanation?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-005::S02"]
        ),
        "rationale": (
            "Tests separation of verified, reported, unknown, and "
            "rule-based information."
        ),
    },
    {
        "query_id": "RAG-EVAL-012",
        "query_family": "AUDIT_FIELDS",
        "query_text": (
            "Which fields must be recorded to make a triage recommendation "
            "auditable?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-005::S03"]
        ),
        "rationale": (
            "Tests required decision-explanation and audit fields."
        ),
    },
    {
        "query_id": "RAG-EVAL-013",
        "query_family": "DISPOSITION_WORDING",
        "query_text": (
            "How should the system express a recommended decline without "
            "presenting it as a final customer decision?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-005::S04"]
        ),
        "rationale": (
            "Tests safe disposition wording and non-finality."
        ),
    },
    {
        "query_id": "RAG-EVAL-014",
        "query_family": "TRACEABILITY",
        "query_text": (
            "What references should be retained for traceability while "
            "avoiding unnecessary customer information?"
        ),
        "relevant_chunk_ids_json": json.dumps(
            ["KB-005::S05"]
        ),
        "rationale": (
            "Tests traceability references and data-minimisation guidance."
        ),
    },
]

expected_evaluation_set = pd.DataFrame(evaluation_rows)

evaluation_dir.mkdir(parents=True, exist_ok=True)

expected_csv = expected_evaluation_set.to_csv(
    index=False,
    lineterminator="\n",
)

expected_hash = hashlib.sha256(
    expected_csv.encode("utf-8")
).hexdigest()

if evaluation_path.exists():
    retrieval_evaluation_set = pd.read_csv(
        evaluation_path,
        dtype=str,
    )

    existing_csv = retrieval_evaluation_set.to_csv(
        index=False,
        lineterminator="\n",
    )

    existing_hash = hashlib.sha256(
        existing_csv.encode("utf-8")
    ).hexdigest()

    if existing_hash != expected_hash:
        raise ValueError(
            "The existing v1 retrieval evaluation set differs from the "
            "frozen definition. Create a new version rather than overwrite it."
        )

    print("Existing frozen evaluation set validated.")
else:
    expected_evaluation_set.to_csv(
        evaluation_path,
        index=False,
    )

    retrieval_evaluation_set = expected_evaluation_set.copy()

    print("New frozen evaluation set created.")

print("Evaluation path:", evaluation_path)
print("Evaluation rows:", len(retrieval_evaluation_set))
print("Evaluation dataset SHA-256:", expected_hash)

known_chunk_ids = set(rag_corpus["chunk_id"])

for row in retrieval_evaluation_set.itertuples(index=False):
    relevant_chunk_ids = json.loads(
        row.relevant_chunk_ids_json
    )

    assert relevant_chunk_ids, (
        f"{row.query_id} has no relevant chunk target."
    )

    assert set(relevant_chunk_ids).issubset(
        known_chunk_ids
    ), (
        f"{row.query_id} references an unknown corpus chunk."
    )

assert len(retrieval_evaluation_set) == 14
assert retrieval_evaluation_set["query_id"].is_unique
assert not retrieval_evaluation_set["query_text"].str.contains(
    "CLM-",
    case=False,
    regex=False,
).any()

display(
    retrieval_evaluation_set[
        [
            "query_id",
            "query_family",
            "query_text",
            "relevant_chunk_ids_json",
        ]
    ]
)

print(
    "\nValidation passed: the fixed retrieval evaluation set contains "
    "14 query-to-chunk relevance targets and is separate from the runtime corpus."
)

Existing frozen evaluation set validated.
Evaluation path: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/data/evaluation/retrieval/rag_retrieval_eval_v1.csv
Evaluation rows: 14
Evaluation dataset SHA-256: b4bcf48af36f2c51034b2108d547509ea2df754e47bee2764f7555b1a8a71c89


,query_id,query_family,query_text,relevant_chunk_ids_json
0,RAG-EVAL-001,EVIDENCE_PROFILE,A claimant reports liquid exposure. Which proo...,"[""KB-002::S03""]"
1,RAG-EVAL-002,EVIDENCE_STATUS,What is the correct operational response to a ...,"[""KB-002::S04""]"
2,RAG-EVAL-003,EVIDENCE_CONTRADICTION,A repair document identifies a different hands...,"[""KB-002::S05""]"
3,RAG-EVAL-004,EVIDENCE_PRINCIPLE,Can a supporting repair quotation make an othe...,"[""KB-002::S02""]"
4,RAG-EVAL-005,MANUAL_REVIEW_PACKET,What details must accompany a case assigned fo...,"[""KB-004::S03""]"
5,RAG-EVAL-006,MANUAL_REVIEW_WRITING,How should an escalation note describe conflic...,"[""KB-004::S04""]"
6,RAG-EVAL-007,MANUAL_REVIEW_SCOPE,What investigations or approvals are outside t...,"[""KB-004::S05""]"
7,RAG-EVAL-008,SOURCE_PRECEDENCE,"When structured sources disagree, which source...","[""KB-001::S03""]"
8,RAG-EVAL-009,MATERIAL_FOLLOW_UP,Should the system request additional paperwork...,"[""KB-001::S05""]"
9,RAG-EVAL-010,TRIAGE_FLOW,What operational sequence should be completed ...,"[""KB-001::S04""]"



Validation passed: the fixed retrieval evaluation set contains 14 query-to-chunk relevance targets and is separate from the runtime corpus.


In [13]:
# ============================================================
# Step 7C: Freeze Retrieval Evaluation Manifest
# ============================================================

import hashlib
import json

manifest_path = (
    project_root
    / "data"
    / "evaluation"
    / "retrieval"
    / "rag_retrieval_eval_v1_manifest.json"
)

evaluation_csv_bytes = evaluation_path.read_bytes()

evaluation_manifest = {
    "evaluation_name": "rag_retrieval_eval_v1",
    "purpose": (
        "Offline evaluation of analyst-guidance retrieval quality only. "
        "This dataset must not be loaded into the runtime retrieval corpus."
    ),
    "evaluation_file": str(
        evaluation_path.relative_to(project_root)
    ),
    "evaluation_file_sha256": hashlib.sha256(
        evaluation_csv_bytes
    ).hexdigest(),
    "query_count": int(len(retrieval_evaluation_set)),
    "relevant_corpus_chunk_count": int(
        len(
            {
                chunk_id
                for value in retrieval_evaluation_set[
                    "relevant_chunk_ids_json"
                ]
                for chunk_id in json.loads(value)
            }
        )
    ),
    "corpus_chunk_count": int(len(rag_corpus)),
    "corpus_fingerprint": semantic_retriever.corpus_fingerprint,
    "approved_document_ids": sorted(
        rag_corpus["document_id"].unique().tolist()
    ),
    "excluded_document_ids": [
        "KB-003",
        "KB-006",
        "KB-007",
    ],
    "primary_metrics": [
        "Hit@1",
        "Hit@3",
        "MRR@3",
    ],
    "retrieval_methods": [
        "lexical_tfidf_v1",
        "semantic_embedding_v1",
        "hybrid_rrf_v1",
    ],
    "semantic_embedding_model": semantic_retriever.embedding_model,
    "semantic_embedding_dimension": (
        semantic_retriever.embedding_dimension
    ),
    "evaluation_boundary": (
        "Results measure retrieval relevance only. They do not measure "
        "claim triage correctness, eligibility, payment, fraud, or "
        "customer communication quality."
    ),
}

manifest_json = json.dumps(
    evaluation_manifest,
    indent=2,
    sort_keys=True,
) + "\n"

if manifest_path.exists():
    existing_manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    if existing_manifest != evaluation_manifest:
        raise ValueError(
            "Existing v1 retrieval-evaluation manifest differs from the "
            "current frozen definition. Create a new version rather than "
            "overwriting it."
        )

    print("Existing frozen manifest validated.")
else:
    manifest_path.write_text(
        manifest_json,
        encoding="utf-8",
    )
    print("New frozen manifest created.")

print("\nManifest path:", manifest_path)
print("\nManifest contents:")
print(manifest_json)

assert evaluation_manifest["query_count"] == 14
assert evaluation_manifest["corpus_chunk_count"] == 21
assert (
    evaluation_manifest["corpus_fingerprint"]
    == semantic_retriever.corpus_fingerprint
)
assert evaluation_manifest["evaluation_file_sha256"] == (
    "b4bcf48af36f2c51034b2108d547509ea2df754e47bee2764f7555b1a8a71c89"
)

print(
    "\nValidation passed: the retrieval evaluation set is bound to the "
    "current approved corpus and evaluation configuration."
)

Existing frozen manifest validated.

Manifest path: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage/data/evaluation/retrieval/rag_retrieval_eval_v1_manifest.json

Manifest contents:
{
  "approved_document_ids": [
    "KB-001",
    "KB-002",
    "KB-004",
    "KB-005"
  ],
  "corpus_chunk_count": 21,
  "corpus_fingerprint": "f73f827d3614e2987a7e976c687043f3fe9e2f5faeeb9c6555508089230ebf48",
  "evaluation_boundary": "Results measure retrieval relevance only. They do not measure claim triage correctness, eligibility, payment, fraud, or customer communication quality.",
  "evaluation_file": "data/evaluation/retrieval/rag_retrieval_eval_v1.csv",
  "evaluation_file_sha256": "b4bcf48af36f2c51034b2108d547509ea2df754e47bee2764f7555b1a8a71c89",
  "evaluation_name": "rag_retrieval_eval_v1",
  "excluded_document_ids": [
    "KB-003",
    "KB-006",
    "KB-007"
  ],
  "primary_metrics": [
    "Hit@1",
    "Hit@3",
    "MRR@3"
  ],
  "purpose": "Offline evalua

In [14]:
# ============================================================
# Step 6D: Hybrid Corpus-Identity Validation
# ============================================================

import importlib

import src.rag.lexical_retriever as lexical_module
import src.rag.semantic_retriever as semantic_module
import src.rag.hybrid_retriever as hybrid_module

importlib.reload(lexical_module)
importlib.reload(semantic_module)
importlib.reload(hybrid_module)

ControlledLexicalRetriever = (
    lexical_module.ControlledLexicalRetriever
)

ControlledSemanticRetriever = (
    semantic_module.ControlledSemanticRetriever
)

ControlledHybridRetriever = (
    hybrid_module.ControlledHybridRetriever
)

lexical_retriever = ControlledLexicalRetriever(rag_corpus)

semantic_retriever = ControlledSemanticRetriever.from_openai(
    corpus=rag_corpus,
    embedding_model="text-embedding-3-small",
    client=openai_client,
)

hybrid_retriever = ControlledHybridRetriever(
    lexical_retriever=lexical_retriever,
    semantic_retriever=semantic_retriever,
)

print(
    "Lexical fingerprint:",
    lexical_retriever.corpus_fingerprint,
)

print(
    "Semantic fingerprint:",
    semantic_retriever.corpus_fingerprint,
)

print(
    "Hybrid fingerprint:",
    hybrid_retriever.corpus_fingerprint,
)

assert (
    lexical_retriever.corpus_fingerprint
    == semantic_retriever.corpus_fingerprint
)

assert (
    hybrid_retriever.corpus_fingerprint
    == semantic_retriever.corpus_fingerprint
)

assert hybrid_retriever.corpus_size == 21

print(
    "\nValidation passed: lexical, semantic, and hybrid retrieval "
    "operate over the identical approved corpus."
)

Lexical fingerprint: f73f827d3614e2987a7e976c687043f3fe9e2f5faeeb9c6555508089230ebf48
Semantic fingerprint: f73f827d3614e2987a7e976c687043f3fe9e2f5faeeb9c6555508089230ebf48
Hybrid fingerprint: f73f827d3614e2987a7e976c687043f3fe9e2f5faeeb9c6555508089230ebf48

Validation passed: lexical, semantic, and hybrid retrieval operate over the identical approved corpus.


## Step 8C — Compare Lexical, Semantic, and Hybrid Retrieval

This step evaluates the three retrieval methods against the frozen v1 retrieval evaluation set.

Metrics measure retrieval relevance only:

- Hit@1: relevant chunk returned first
- Hit@3: relevant chunk returned within the top three
- MRR@3: rewards a relevant chunk appearing earlier in the ranking
- No-result rate: queries for which no chunks were returned

The results do not measure triage correctness, policy eligibility, customer communication quality, or claim outcome quality.

In [15]:
# ============================================================
# Step 8C: Live Retrieval Evaluation Comparison
# ============================================================

from src.rag.evaluation import (
    evaluate_retrieval_method,
    load_retrieval_evaluation_set,
)

evaluation_top_k = 3

retrieval_evaluation_set = load_retrieval_evaluation_set(
    evaluation_path=evaluation_path,
    known_chunk_ids=rag_corpus["chunk_id"].tolist(),
)

known_chunk_ids = rag_corpus["chunk_id"].tolist()

print("Evaluation queries:", len(retrieval_evaluation_set))
print("Evaluation top_k:", evaluation_top_k)


def retrieve_lexical(query_text: str, top_k: int):
    return lexical_retriever.retrieve(
        query=query_text,
        top_k=top_k,
    )


def retrieve_semantic(query_text: str, top_k: int):
    return semantic_retriever.retrieve(
        query=query_text,
        top_k=top_k,
        min_relevance_score=0.20,
        client=openai_client,
    )


def retrieve_hybrid(query_text: str, top_k: int):
    return hybrid_retriever.retrieve(
        query=query_text,
        top_k=top_k,
        candidate_k=max(5, top_k),
        min_semantic_relevance_score=0.20,
        semantic_client=openai_client,
    )


lexical_evaluation = evaluate_retrieval_method(
    evaluation_set=retrieval_evaluation_set,
    known_chunk_ids=known_chunk_ids,
    method_name="Lexical TF-IDF",
    retrieve_query=retrieve_lexical,
    top_k=evaluation_top_k,
)

semantic_evaluation = evaluate_retrieval_method(
    evaluation_set=retrieval_evaluation_set,
    known_chunk_ids=known_chunk_ids,
    method_name="Semantic Embedding",
    retrieve_query=retrieve_semantic,
    top_k=evaluation_top_k,
)

hybrid_evaluation = evaluate_retrieval_method(
    evaluation_set=retrieval_evaluation_set,
    known_chunk_ids=known_chunk_ids,
    method_name="Hybrid RRF",
    retrieve_query=retrieve_hybrid,
    top_k=evaluation_top_k,
)

evaluation_runs = [
    lexical_evaluation,
    semantic_evaluation,
    hybrid_evaluation,
]

summary_metrics = pd.DataFrame(
    [
        run["summary_metrics"]
        for run in evaluation_runs
    ]
)

summary_metrics = summary_metrics[
    [
        "method_name",
        "query_count",
        "top_k",
        "hit_at_1",
        "hit_at_3",
        "mrr_at_3",
        "no_result_rate",
    ]
]

print("\nOverall retrieval metrics:")
display(
    summary_metrics.style.format(
        {
            "hit_at_1": "{:.1%}",
            "hit_at_3": "{:.1%}",
            "mrr_at_3": "{:.3f}",
            "no_result_rate": "{:.1%}",
        }
    )
)

family_metrics = pd.concat(
    [
        run["family_metrics"].assign(
            method_name=run["method_name"]
        )
        for run in evaluation_runs
    ],
    ignore_index=True,
)

family_metrics = family_metrics[
    [
        "method_name",
        "query_family",
        "query_count",
        "hit_at_1",
        "hit_at_3",
        "mrr_at_3",
    ]
]

print("\nMetrics by query family:")
display(
    family_metrics.style.format(
        {
            "hit_at_1": "{:.1%}",
            "hit_at_3": "{:.1%}",
            "mrr_at_3": "{:.3f}",
        }
    )
)

per_query_comparison = pd.concat(
    [
        run["query_results"]
        for run in evaluation_runs
    ],
    ignore_index=True,
)

per_query_comparison = per_query_comparison[
    [
        "method_name",
        "query_id",
        "query_family",
        "relevant_chunk_ids",
        "retrieved_chunk_ids",
        "first_relevant_rank",
        "hit_at_1",
        "hit_at_3",
        "mrr_at_3",
        "retrieval_status",
    ]
]

print("\nPer-query comparison:")
display(per_query_comparison)

misses = per_query_comparison.loc[
    per_query_comparison["hit_at_3"].eq(0)
].copy()

print("\nTop-3 misses requiring review:")
if misses.empty:
    print("No top-3 misses.")
else:
    display(
        misses[
            [
                "method_name",
                "query_id",
                "query_family",
                "relevant_chunk_ids",
                "retrieved_chunk_ids",
                "retrieval_status",
            ]
        ]
    )

assert len(retrieval_evaluation_set) == 14
assert set(summary_metrics["method_name"]) == {
    "Lexical TF-IDF",
    "Semantic Embedding",
    "Hybrid RRF",
}
assert summary_metrics["query_count"].eq(14).all()

print(
    "\nEvaluation completed: all three methods were measured against "
    "the same frozen v1 retrieval set and approved corpus."
)

Evaluation queries: 14
Evaluation top_k: 3

Overall retrieval metrics:


,method_name,query_count,top_k,hit_at_1,hit_at_3,mrr_at_3,no_result_rate
0,Lexical TF-IDF,14,3,57.1%,85.7%,0.702,0.0%
1,Semantic Embedding,14,3,78.6%,92.9%,0.857,0.0%
2,Hybrid RRF,14,3,71.4%,92.9%,0.798,0.0%



Metrics by query family:


,method_name,query_family,query_count,hit_at_1,hit_at_3,mrr_at_3
0,Lexical TF-IDF,AUDIT_FIELDS,1,0.0%,100.0%,0.333
1,Lexical TF-IDF,DISPOSITION_WORDING,1,100.0%,100.0%,1.000
2,Lexical TF-IDF,EVIDENCE_CONTRADICTION,1,100.0%,100.0%,1.000
3,Lexical TF-IDF,EVIDENCE_PRINCIPLE,1,0.0%,0.0%,0.000
4,Lexical TF-IDF,EVIDENCE_PROFILE,1,100.0%,100.0%,1.000
5,Lexical TF-IDF,EVIDENCE_STATUS,1,100.0%,100.0%,1.000
6,Lexical TF-IDF,EXPLANATION_FACT_TYPES,1,100.0%,100.0%,1.000
7,Lexical TF-IDF,MANUAL_REVIEW_PACKET,1,0.0%,0.0%,0.000
8,Lexical TF-IDF,MANUAL_REVIEW_SCOPE,1,0.0%,100.0%,0.500
9,Lexical TF-IDF,MANUAL_REVIEW_WRITING,1,100.0%,100.0%,1.000



Per-query comparison:


,method_name,query_id,query_family,relevant_chunk_ids,retrieved_chunk_ids,first_relevant_rank,hit_at_1,hit_at_3,mrr_at_3,retrieval_status
0,Lexical TF-IDF,RAG-EVAL-001,EVIDENCE_PROFILE,[KB-002::S03],"[KB-002::S03, KB-005::S05, KB-002::S02]",1.0,1,1,1.000000,RESULTS_FOUND
1,Lexical TF-IDF,RAG-EVAL-002,EVIDENCE_STATUS,[KB-002::S04],"[KB-002::S04, KB-001::S01, KB-002::S02]",1.0,1,1,1.000000,RESULTS_FOUND
2,Lexical TF-IDF,RAG-EVAL-003,EVIDENCE_CONTRADICTION,[KB-002::S05],"[KB-002::S05, KB-004::S02, KB-001::S04]",1.0,1,1,1.000000,RESULTS_FOUND
3,Lexical TF-IDF,RAG-EVAL-004,EVIDENCE_PRINCIPLE,[KB-002::S02],"[KB-002::S05, KB-002::S03, KB-001::S02]",NaN,0,0,0.000000,RESULTS_FOUND
4,Lexical TF-IDF,RAG-EVAL-005,MANUAL_REVIEW_PACKET,[KB-004::S03],"[KB-002::S02, KB-005::S04, KB-001::S02]",NaN,0,0,0.000000,RESULTS_FOUND
5,Lexical TF-IDF,RAG-EVAL-006,MANUAL_REVIEW_WRITING,[KB-004::S04],"[KB-004::S04, KB-004::S02, KB-001::S04]",1.0,1,1,1.000000,RESULTS_FOUND
6,Lexical TF-IDF,RAG-EVAL-007,MANUAL_REVIEW_SCOPE,[KB-004::S05],"[KB-005::S04, KB-004::S05, KB-001::S01]",2.0,0,1,0.500000,RESULTS_FOUND
7,Lexical TF-IDF,RAG-EVAL-008,SOURCE_PRECEDENCE,[KB-001::S03],"[KB-001::S02, KB-001::S03, KB-005::S02]",2.0,0,1,0.500000,RESULTS_FOUND
8,Lexical TF-IDF,RAG-EVAL-009,MATERIAL_FOLLOW_UP,[KB-001::S05],"[KB-001::S05, KB-001::S04, KB-002::S04]",1.0,1,1,1.000000,RESULTS_FOUND
9,Lexical TF-IDF,RAG-EVAL-010,TRIAGE_FLOW,[KB-001::S04],"[KB-001::S01, KB-001::S04, KB-001::S03]",2.0,0,1,0.500000,RESULTS_FOUND



Top-3 misses requiring review:


,method_name,query_id,query_family,relevant_chunk_ids,retrieved_chunk_ids,retrieval_status
3,Lexical TF-IDF,RAG-EVAL-004,EVIDENCE_PRINCIPLE,[KB-002::S02],"[KB-002::S05, KB-002::S03, KB-001::S02]",RESULTS_FOUND
4,Lexical TF-IDF,RAG-EVAL-005,MANUAL_REVIEW_PACKET,[KB-004::S03],"[KB-002::S02, KB-005::S04, KB-001::S02]",RESULTS_FOUND
25,Semantic Embedding,RAG-EVAL-012,AUDIT_FIELDS,[KB-005::S03],"[KB-001::S04, KB-005::S01, KB-001::S02]",RESULTS_FOUND
39,Hybrid RRF,RAG-EVAL-012,AUDIT_FIELDS,[KB-005::S03],"[KB-001::S02, KB-005::S01, KB-001::S04]",RESULTS_FOUND



Evaluation completed: all three methods were measured against the same frozen v1 retrieval set and approved corpus.


In [16]:
from src.rag.controlled_query_builder import AuthoritativeTriageFacts

facts = AuthoritativeTriageFacts(
    claim_type="ACCIDENTAL_DAMAGE",
    incident_category="SCREEN_DAMAGE",
    coverage_outcome="ELIGIBLE",
    evidence_state="MISSING_REQUIRED",
    manual_review_required=True,
    product_family="SMARTPHONE",
    required_evidence_codes=(
        "DAMAGE_PHOTOS",
        "PROOF_OF_PURCHASE",
    ),
    manual_review_reason_codes=(
        "SERIAL_MISMATCH",
    ),
)

print(facts)

AuthoritativeTriageFacts(claim_type='ACCIDENTAL_DAMAGE', incident_category='SCREEN_DAMAGE', coverage_outcome='ELIGIBLE', evidence_state='MISSING_REQUIRED', manual_review_required=True, product_family='SMARTPHONE', required_evidence_codes=('DAMAGE_PHOTOS', 'PROOF_OF_PURCHASE'), manual_review_reason_codes=('SERIAL_MISMATCH',))


In [17]:
import importlib
import src.rag.controlled_query_builder as cqb

cqb = importlib.reload(cqb)

facts_a = cqb.AuthoritativeTriageFacts(
    claim_type="ACCIDENTAL_DAMAGE",
    incident_category="SCREEN_DAMAGE",
    coverage_outcome="ELIGIBLE",
    evidence_state="MISSING_REQUIRED",
    manual_review_required=True,
    product_family="SMARTPHONE",
    required_evidence_codes=(
        "DAMAGE_PHOTOS",
        "PROOF_OF_PURCHASE",
    ),
    manual_review_reason_codes=(
        "SERIAL_MISMATCH",
    ),
)

# Same facts, deliberately different evidence-code order.
facts_b = cqb.AuthoritativeTriageFacts(
    claim_type="ACCIDENTAL_DAMAGE",
    incident_category="SCREEN_DAMAGE",
    coverage_outcome="ELIGIBLE",
    evidence_state="MISSING_REQUIRED",
    manual_review_required=True,
    product_family="SMARTPHONE",
    required_evidence_codes=(
        "PROOF_OF_PURCHASE",
        "DAMAGE_PHOTOS",
    ),
    manual_review_reason_codes=(
        "SERIAL_MISMATCH",
    ),
)

builder = cqb.ControlledRAGQueryBuilder()

query_a = builder.build(facts_a)
query_b = builder.build(facts_b)

assert query_a.query_text == query_b.query_text
assert query_a.query_fingerprint == query_b.query_fingerprint
assert query_a.audience == "analyst"
assert query_a.authority == "non_authoritative_guidance"

print(query_a.query_text)
print("\nFingerprint:", query_a.query_fingerprint)
print("Source:", query_a.source)
print("Audience:", query_a.audience)
print("Authority:", query_a.authority)

Device protection claim analyst guidance. Claim type: ACCIDENTAL_DAMAGE (accidental damage). Incident category: SCREEN_DAMAGE (screen damage). Coverage outcome: ELIGIBLE (eligible). Evidence state: MISSING_REQUIRED (missing required). Product family: SMARTPHONE (smartphone). Required evidence codes: DAMAGE_PHOTOS (damage photos), PROOF_OF_PURCHASE (proof of purchase). Manual review required: YES. Manual-review reason codes: SERIAL_MISMATCH (serial mismatch). Retrieve approved internal knowledge-base guidance for analyst evidence handling, documentation standards, manual-review procedures, and appropriate analyst next steps.

Fingerprint: 59b054369b21c6cda4a016708a6b512ad5555aee941788b5c1ded033a909abef
Source: authoritative_deterministic_triage_facts
Audience: analyst
Authority: non_authoritative_guidance
